# Train & Fine-tune DistilBERT — Sentiment Classification
Simple, linear notebook: split → tokenize → train → evaluate → save → predict.


# mount drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 1. Imports

In [2]:
import os
import time
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [3]:
os.chdir('/content/drive/MyDrive/Ironhack_projects/project_3')
print(os.getcwd())
#print(os.listdir())

/content/drive/MyDrive/Ironhack_projects/project_3


## 2. Settings


In [4]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

MODEL_OUTPUT_DIR = f"models/{MODEL_NAME}"
#LOG_FILE = "logs/experiment_log.csv"          # every run gets appended here as one row

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
#os.makedirs("logs", exist_ok=True)

## 3. Load the balanced dataset

In [5]:
drive_path = "/content/drive/MyDrive/Ironhack_projects/datasets/reviews_balanced.csv"
local_path = "../data/processed/reviews_balanced.csv"

df = pd.read_csv(drive_path)
#df = df[['reviews.text', 'reviews.rating']]
#df = df.dropna(subset=["clean_text", "label"])
print(df.shape)
df["label"].value_counts()

(5982, 10)


,count
label,
positive,3000
negative,1653
neutral,1329


## 4. Map text labels to numbers
Transformers need integer labels, not strings.

In [6]:
label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {0: "negative", 1: "neutral", 2: "positive"}

df["labels"] = df["label"].map(label2id)
df[["clean_text", "label", "labels"]].head()

,clean_text,label,labels
0,very little capacity. they last only about 25 ...,negative,0
1,my daughter just loved this kindle fire. she s...,positive,2
2,amazon batteries are always a great deal!,positive,2
3,amazon fire is a good tablet if you like e-boo...,negative,0
4,great tablet for my son. live the endless libr...,positive,2


## 5. Split into train / validation / test (80 / 10 / 10)


In [7]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df["labels"], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["labels"], random_state=42
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

# train_df.to_csv("data/train.csv", index=False)
# val_df.to_csv("data/val.csv", index=False)
# test_df.to_csv("data/test.csv", index=False)

Train: (4785, 11) Val: (598, 11) Test: (599, 11)


## 6. Check class balance
The dataset is undersampled due to extreme class imbalance (refer preprocessing noteboook). We handle the imbalance again by using **class weights**, so every review from every split can still be used for training.

In [8]:
print("Train labels:\n", train_df["label"].value_counts())
print("\nVal labels:\n", val_df["label"].value_counts())
print("\nTest labels:\n", test_df["label"].value_counts())

Train labels:
 label
positive    2400
negative    1322
neutral     1063
Name: count, dtype: int64

Val labels:
 label
positive    300
negative    165
neutral     133
Name: count, dtype: int64

Test labels:
 label
positive    300
negative    166
neutral     133
Name: count, dtype: int64


## 6.1. Compute class weights to handle imbalance
`sklearn`'s `compute_class_weight("balanced", ...)` gives each class a weight inversely proportional to its frequency, so the rare `negative`/`neutral` classes contribute as much to the loss as the dominant `positive` class. These weights get passed into a custom `Trainer` further down — no resampling of the data needed.

In [9]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1, 2]),
    y=train_df["labels"],
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights (negative, neutral, positive):", class_weights)

Class weights (negative, neutral, positive): tensor([1.2065, 1.5005, 0.6646])


## 7. Tokenize the text
Convert each split into a HuggingFace `Dataset` and tokenize with the DistilBERT tokenizer.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

col_review = "clean_text" # 'reviews.text' # "clean_text"

def tokenize(batch):
    return tokenizer(batch[col_review], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[[col_review, "labels"]])
val_ds   = Dataset.from_pandas(val_df[[col_review, "labels"]])
test_ds  = Dataset.from_pandas(test_df[[col_review, "labels"]])

train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=keep_cols)
val_ds.set_format(type="torch", columns=keep_cols)
test_ds.set_format(type="torch", columns=keep_cols)

train_ds

Map:   0%|          | 0/4785 [00:00<?, ? examples/s]

Map:   0%|          | 0/598 [00:00<?, ? examples/s]

Map:   0%|          | 0/599 [00:00<?, ? examples/s]

Dataset({
    features: ['clean_text', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 4785
})

## 8. Load the model

In [11]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 9. Define the evaluation metric function

In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1_macro": f1}

## 10. Set training arguments and fine-tune (with class-weighted loss)

In [13]:
class WeightedTrainer(Trainer):
    """Trainer that applies class weights inside the loss function, so
    minority classes (negative/neutral) count as much as the majority
    (positive) class during training."""
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
    save_total_limit=2,
)

trainer = WeightedTrainer(
    class_weights=class_weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

In [14]:
start_time = time.time()
trainer.train()
train_time_sec = time.time() - start_time

print(f"Training took {train_time_sec/60:.1f} minutes")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1 Macro
1,0.576951,0.568119,0.806020,0.776396,0.776088,0.775166
2,0.366085,0.511567,0.831104,0.801056,0.811972,0.805721
3,0.360887,0.519115,0.841137,0.815444,0.821069,0.817324


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training took 3.1 minutes


## 11. Evaluate on the held-out test set

In [15]:
predictions = trainer.predict(test_ds)
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"]))

cm = confusion_matrix(y_true, y_pred)
print("Confusion matrix:\n", cm)

test_metrics = predictions.metrics
test_metrics

              precision    recall  f1-score   support

    negative       0.81      0.81      0.81       166
     neutral       0.64      0.74      0.68       133
    positive       0.95      0.88      0.91       300

    accuracy                           0.83       599
   macro avg       0.80      0.81      0.80       599
weighted avg       0.84      0.83      0.83       599

Confusion matrix:
 [[135  27   4]
 [ 25  98  10]
 [  7  29 264]]


{'test_loss': 0.5006601810455322,
 'test_accuracy': 0.8297161936560935,
 'test_precision': 0.7981290525554515,
 'test_recall': 0.8100317057704501,
 'test_f1_macro': 0.8024108165892283,
 'test_runtime': 2.0786,
 'test_samples_per_second': 288.178,
 'test_steps_per_second': 9.141}

## 12. Quantize the model to shrink it for deployment

**Dynamic quantization** converts the weights of every `Linear` layer from float32 to int8 after training (no retraining needed), typically shrinking the model ~4x and speeding up CPU inference — a good fit for serving predictions from a FastAPI backend on CPU. The tradeoff is a small, usually negligible, drop in accuracy, which we check in Section 13d below.

In [24]:
QUANTIZED_MODEL_DIR = f"models/{MODEL_NAME}-quantized"
os.makedirs(QUANTIZED_MODEL_DIR, exist_ok=True)

# Reload the saved fp32 model fresh, on CPU — dynamic quantization
# only targets CPU inference
# fp32_model = AutoModelForSequenceClassification.from_pretrained(MODEL_OUTPUT_DIR)
# fp32_model.eval()
#fp32_model = trainer.model

model_q = trainer.model.cpu()
model_q.eval()

# Quantization
quantized_model = torch.quantization.quantize_dynamic(
    model_q, {nn.Linear}, dtype=torch.qint8
)

/tmp/ipykernel_21598/85555117.py:14: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


## 13. Compare model size
Compare the size of original model and quantized model.

In [25]:
def get_model_size(model):
    #model = trainer.model

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    param_size = sum(p.numel() * p.element_size() for p in model.parameters())

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Model parameter size: {param_size / (1024 ** 2):.2f} MB")

In [26]:
get_model_size(trainer.model)

Total parameters: 66,955,779
Trainable parameters: 66,955,779
Model parameter size: 255.42 MB


In [27]:
get_model_size(quantized_model)

Total parameters: 23,854,080
Trainable parameters: 23,854,080
Model parameter size: 91.00 MB


## 14. Save the quantized model and tokenizer

In [ ]:
# trainer.save_model(MODEL_OUTPUT_DIR)
# tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
# print("Saved tokenizer to", MODEL_OUTPUT_DIR)

In [28]:
# Saving the whole model object directly with torch.save is the simplest
# reliable way to persist it.
torch.save(quantized_model, os.path.join(QUANTIZED_MODEL_DIR, "quantized_model.pt"))
tokenizer.save_pretrained(QUANTIZED_MODEL_DIR)   # tokenizer is unaffected by quantization

print("Saved quantized model to", QUANTIZED_MODEL_DIR)

Saved quantized model to models/distilbert-base-uncased-quantized


## 15. Sanity-check accuracy after quantization
Run the quantized model over the held-out test set and confirm metrics are still close to the fp32 model's results from Section 11.

In [29]:
quantized_model.eval()
#quantized_model.to("cpu")  # dynamic quantization only runs on CPU

all_preds, all_labels = [], []
eval_batch_size = BATCH_SIZE * 2

with torch.no_grad():
    for i in range(0, len(test_df), eval_batch_size):
        batch = test_df.iloc[i:i + eval_batch_size]
        inputs = tokenizer(
            batch[col_review].tolist(), truncation=True, padding=True,
            max_length=MAX_LENGTH, return_tensors="pt"
        )
        logits = quantized_model(**inputs).logits
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(batch["labels"].tolist())

print(classification_report(all_labels, all_preds, target_names=["negative", "neutral", "positive"]))

              precision    recall  f1-score   support

    negative       0.80      0.83      0.82       166
     neutral       0.65      0.72      0.69       133
    positive       0.94      0.88      0.91       300

    accuracy                           0.83       599
   macro avg       0.80      0.81      0.80       599
weighted avg       0.84      0.83      0.83       599



- The quantized model performs almost same as the original model.

## 16. Load the saved QUANTIZED model back and predict on new text
This is what your FastAPI backend should do at startup: load the small quantized model once (CPU), then reuse it for every prediction request.

In [30]:
device = "cpu"  # dynamically quantized models run on CPU

loaded_tokenizer = AutoTokenizer.from_pretrained(QUANTIZED_MODEL_DIR)
loaded_model = torch.load(
    os.path.join(QUANTIZED_MODEL_DIR, "quantized_model.pt"),
    map_location=device,
    weights_only=False,   # this is a full quantized nn.Module, not just a state_dict
)
loaded_model.eval()

new_reviews = [
    "This product broke after two days, extremely disappointed.",
    "Amazing quality for the price, works exactly as described!",
    "It's okay, does the job but nothing special.",
]

inputs = loaded_tokenizer(new_reviews, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt")

with torch.no_grad():
    logits = loaded_model(**inputs).logits
    probs = torch.softmax(logits, dim=1).cpu().numpy()
    preds = probs.argmax(axis=1)

for text, pred, prob in zip(new_reviews, preds, probs):
    print(f"{id2label[pred].upper():9s} | conf={prob.max():.3f} | {text}")

/usr/local/lib/python3.12/dist-packages/torch/_utils.py:455: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


NEGATIVE  | conf=0.967 | This product broke after two days, extremely disappointed.
POSITIVE  | conf=0.974 | Amazing quality for the price, works exactly as described!
NEUTRAL   | conf=0.736 | It's okay, does the job but nothing special.


# Notes

- Used the **preprocessed and undersampled dataset** for model training. **Class weights** were additionally applied during training to address the remaining class imbalance.
- Model performance improved after explicitly handling the **class imbalance**, indicating that class weighting helped the model better learn the underrepresented classes.
- Fine-tuned the **DistilBERT-base-uncased** model for the classification task. However, the resulting model size was approximately **225 MB**, which is relatively large and may not be suitable for deployment due to size limitations.
- Applied **dynamic quantization** to reduce the model size. The quantized model size was reduced from approximately **225 MB to 91 MB**, while maintaining **similar model performance**.
- Therefore, the **quantized DistilBERT model** was selected as the final model for deployment, providing a good balance between **model size and predictive performance**.